# Lab 5: CIFAR-10 Transfer Learning
AI Lab, University of Technology Bahrain. About 20 minutes. GPU (required).

**You will learn**
- Colour images: 32×32×3, 10 everyday classes
- Reusing an ImageNet-trained network (MobileNetV2)
- Feature extraction first, then fine-tuning

**Before you run**
- On Kaggle: right-hand panel, *Settings → Accelerator → GPU T4 x2* (when the lab needs a GPU) and *Settings → Internet → On*. GPU and Internet need a phone-verified account.
- On Colab: *Runtime → Change runtime type → T4 GPU*.
- Run the cells from top to bottom with **Shift + Enter**.

In [ ]:
# Setup: run this cell first
import os, sys, glob, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
OUT_DIR = "/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else "."
gpus = tf.config.list_physical_devices("GPU")
print("Platform:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local")
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", [g.name for g in gpus] if gpus else "none, running on CPU. See 'Before you run' above to switch a GPU on")
keras.utils.set_random_seed(42)

In [ ]:
# Dataset loader: downloads with Keras; on Kaggle it can also use a copy attached with "+ Add Input"
def load_dataset(name):
    """name: 'mnist', 'fashion_mnist' or 'cifar10' -> (x_train, y_train), (x_test, y_test)"""
    try:
        return getattr(keras.datasets, name).load_data()
    except Exception as err:
        if name == "mnist":  # Kaggle-attached copy of mnist.npz, if any
            hits = glob.glob("/kaggle/input/**/mnist.npz", recursive=True)
            if hits:
                d = np.load(hits[0])
                return (d["x_train"], d["y_train"]), (d["x_test"], d["y_test"])
        raise RuntimeError(
            "The dataset did not download. On Kaggle, switch on Settings > Internet "
            "(needs a phone-verified account). On Colab, check your connection. Then run this cell again.") from err

In [ ]:
(x_train, y_train), (x_test, y_test) = load_dataset("cifar10")
y_train, y_test = y_train.ravel(), y_test.ravel()
CLASSES = ["airplane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for ax, i in zip(axes.flat, range(20)):
    ax.imshow(x_train[i]); ax.set_title(CLASSES[y_train[i]], fontsize=8); ax.axis("off")
plt.show()

## 1. Borrow a pretrained backbone
MobileNetV2 learned general visual features from 1.2 M ImageNet photos. We **freeze** it and only train a small new head.

In [ ]:
IMG = 96  # upscale 32 -> 96 so the pretrained filters see objects at a familiar size
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=(32, 32, 3))
x = layers.Resizing(IMG, IMG)(inputs)
x = layers.RandomFlip("horizontal")(x)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(10, activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"Trainable params: {sum(np.prod(w.shape) for w in model.trainable_weights):,}  "
      f"of {model.count_params():,}")

In [ ]:
t = time.time()
history = model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
print(f"Stage 1 (frozen) took {time.time()-t:.0f}s, test acc:", round(model.evaluate(x_test, y_test, verbose=0)[1], 4))

## 2. Fine-tune the top of the backbone
Unfreeze the last ~30 layers and continue with a **much smaller** learning rate so we don't destroy the pretrained features.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
print("Stage 2 (fine-tuned) test acc:", round(model.evaluate(x_test, y_test, verbose=0)[1], 4))

In [ ]:
# Learning curves
def plot_history(h):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(h.history["loss"], label="train"); ax[0].plot(h.history["val_loss"], label="validation")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h.history["accuracy"], label="train"); ax[1].plot(h.history["val_accuracy"], label="validation")
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
plot_history(history)

In [ ]:
idx = np.random.default_rng(1).choice(len(x_test), 12, replace=False)
pred = model.predict(x_test[idx], verbose=0).argmax(1)
fig, axes = plt.subplots(1, 12, figsize=(15, 2))
for ax, i, p in zip(axes, idx, pred):
    ax.imshow(x_test[i]); ax.axis("off")
    ax.set_title(CLASSES[p], fontsize=8, color="green" if p == y_test[i] else "crimson")
plt.show()

## Try it
1. Train the same head **without** pretrained weights (`weights=None`). How far does accuracy drop in 3 epochs?
2. Try `IMG = 64` and `IMG = 128`. What happens to speed and accuracy?
3. Swap MobileNetV2 for `keras.applications.EfficientNetB0` (remember its own `preprocess_input`).

### Save your work
- **Kaggle:** *Save Version → Quick Save* keeps your edits; *Save & Run All* also keeps the outputs. Files written to `/kaggle/working` appear under **Output**.
- **Colab:** *File → Save a copy in Drive*. Files in `/content` are deleted when the session ends, so download anything you need from the Files panel first.